In [1]:
# Import necessary libraries
import os
import whisper
import spacy
import json
import faiss
import numpy as np
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from starlette.responses import JSONResponse
from typing import List
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sentence_transformers import SentenceTransformer

# Load models
asr_model = whisper.load_model("base")
nlp = spacy.load("en_core_web_sm")  # Updated to use en_core_web_sm
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# FastAPI app
app = FastAPI()

# Dummy database of past notes
past_notes = [
    "Patient with fever and cough. Diagnosed with viral infection. Treated with paracetamol.",
    "Shortness of breath and wheezing. Diagnosed with asthma exacerbation. Treated with inhalers.",
    "Abdominal pain and vomiting. Suspected gastroenteritis. Given IV fluids and antiemetics.",
    "Chest pain radiating to left arm. ECG showed ST elevation. Immediate PCI performed.",
    "High-grade fever and rash. Suspected dengue. Platelet count monitored. IV fluids administered.",
    "Head trauma after fall. CT scan negative. Observed in ER for 6 hours.",
    "Persistent diarrhea. Suspected food poisoning. Stool test ordered. Advised hydration.",
    "Lower back pain. No neurological deficits. Prescribed NSAIDs.",
    "Shortness of breath on exertion. ECG normal. Echo scheduled. Suspect heart failure.",
    "Sudden onset of confusion. Suspect stroke. CT and MRI ordered. tPA administered."
]

# Topic modeling using LDA
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(past_notes)
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_model.fit(X)

def get_topics(model, feature_names, n_top_words=5):
    topics = []
    for topic_idx, topic in enumerate(model.components_):
        words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics.append(words)
    return topics

topics = get_topics(lda_model, vectorizer.get_feature_names_out())

# Prepare FAISS index
past_embeddings = embedding_model.encode(past_notes)
past_embeddings = np.array(past_embeddings).astype("float32")
index = faiss.IndexFlatL2(past_embeddings.shape[1])
index.add(past_embeddings)

SYMPTOMS = {
    "fever", "cough", "shortness of breath", "chest pain",
    "abdominal pain", "vomiting", "diarrhea", "headache",
    "confusion", "wheezing", "fatigue", "dizziness"
}

DISEASES = {
    "asthma", "stroke", "heart failure", "gastroenteritis",
    "food poisoning", "viral infection", "dengue"
}

MEDICATIONS = {
    "paracetamol", "ibuprofen", "aspirin",
    "inhalers", "iv fluids", "tpa", "antiemetics", "nsaids"
}

# Helper functions
def transcribe_audio(file_path):
    result = asr_model.transcribe(file_path)
    return result["text"]

def extract_medical_entities(text: str):
    text_lower = text.lower()

    extracted = []

    for symptom in SYMPTOMS:
        if symptom in text_lower:
            extracted.append({"text": symptom, "label": "SYMPTOM"})

    for disease in DISEASES:
        if disease in text_lower:
            extracted.append({"text": disease, "label": "DISEASE"})

    for med in MEDICATIONS:
        if med in text_lower:
            extracted.append({"text": med, "label": "MEDICATION"})

    return extracted

def retrieve_similar_notes(query_text):
    query_embedding = embedding_model.encode([query_text]).astype("float32")
    distances, indices = index.search(query_embedding, k=3)
    return [past_notes[i] for i in indices[0]]

def generate_structured_note(text, similar_notes):
    return {
        "History": f"{text[:100]}...",
        "Physical Exam": "Normal vitals. Chest clear. No distress.",
        "Medical Decision Making": f"Based on similar past cases: {similar_notes[0]}",
        "Procedure": "None performed at this time."
    }

# Define API models

class MedicalEntity(BaseModel):
    text: str
    label: str
    
class TranscriptionResponse(BaseModel):
    transcription: str
    similar_notes: List[str]
    structured_note: dict
    topics: List[List[str]]

class TextInput(BaseModel):
    text: str

@app.post("/process_audio", response_model=TranscriptionResponse)
async def process_audio(file: UploadFile = File(...)):
    temp_path = f"temp_{file.filename}"
    with open(temp_path, "wb") as f:
        f.write(await file.read())

    text = transcribe_audio(temp_path)
    os.remove(temp_path)

    entities = extract_medical_entities(text)
    similar_notes = retrieve_similar_notes(text)
    structured_note = generate_structured_note(text, similar_notes)

    return JSONResponse(content={
        "transcription": text,
        "medical_entities": entities,
        "similar_notes": similar_notes,
        "structured_note": structured_note,
        "topics": topics
    })

@app.post("/process_text", response_model=TranscriptionResponse)
async def process_text(input: TextInput):
    text = input.text

    entities = extract_medical_entities(text)
    similar_notes = retrieve_similar_notes(text)
    structured_note = generate_structured_note(text, similar_notes)

    return JSONResponse(content={
        "transcription": text,
        "medical_entities": entities,
        "similar_notes": similar_notes,
        "structured_note": structured_note,
        "topics": topics
    })


OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.